# AdaTTT — Tiny-Dataset End-to-End Demo

This notebook is the runnable example required by the report. It demonstrates the full AdaTTT pipeline on a tiny synthetic VQA-shaped dataset so the model, the TTT inner loop, the gate routing, and the Pareto sweep can all be exercised without paying full-dataset compute.

**Stages:**
1. Environment setup
2. Build a tiny synthetic dataset (8 samples, 224x224 RGB images, BERT-shaped tokens)
3. Instantiate `FullVQAModel` with frozen encoders, fusion, gate, and prediction head
4. Run a base forward pass and record gate confidences
5. Run the per-sample TTT inner loop (`TTTAdapter.adapt_and_predict`) and verify save-and-restore
6. Run the adaptive router across a small threshold sweep and plot the accuracy vs. compute trade-off

Run on a local CPU (slow but works) or a Colab T4/H100. The full project lives at: <https://drive.google.com/drive/folders/1Bs9AXg5yVbtT9In8t2FfW4bA9n3mf0J_?usp=share_link>

## 1. Environment setup

Install the package in editable mode if it is not already importable. On Colab, mount the project directory first.

In [ ]:
# If running outside the repo, uncomment the next two lines and adjust the path.
# %cd /content/drive/MyDrive/AdaTTT
# !pip install -q -e .

import sys, os, copy, math, json, random
import numpy as np
import torch
import torch.nn.functional as F

torch.manual_seed(42); np.random.seed(42); random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 2. Tiny synthetic dataset

We build 8 samples with the exact tensor shapes the real loader produces:
- `image`: `(3, 224, 224)` float32
- `input_ids`: `(20,)` int64 BERT token ids
- `attention_mask`: `(20,)` int64 0/1 mask
- `answer_idx`: int in `[0, N)` where `N=10` answer classes for this tiny demo.

We mark four samples as 'easy' (consistent visual statistics) and four as 'hard' (high-noise images) so the gate has something to learn against later.

In [ ]:
N_CLASSES = 10
BATCH = 8
SEQ_LEN = 20

def make_tiny_batch(batch=BATCH, seq_len=SEQ_LEN, n_classes=N_CLASSES):
    images = []
    for i in range(batch):
        if i < batch // 2:
            img = torch.randn(3, 224, 224) * 0.2          # 'easy': low-variance
        else:
            img = torch.randn(3, 224, 224) * 1.5          # 'hard': high-variance
        images.append(img)
    images = torch.stack(images)
    input_ids = torch.randint(low=101, high=29000, size=(batch, seq_len))
    attention_mask = torch.ones((batch, seq_len), dtype=torch.long)
    answer_idx = torch.randint(low=0, high=n_classes, size=(batch,))
    return {
        'image': images,
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'answer_idx': answer_idx,
    }

batch = make_tiny_batch()
for k, v in batch.items():
    print(f'{k:14s} shape={tuple(v.shape)} dtype={v.dtype}')

## 3. Instantiate the model

We build the same components used in the production pipeline: a frozen ViT-B/16, a frozen BERT-base, the trainable bidirectional cross-modal fusion, the confidence gate, and the prediction head. The encoders are downloaded once from Hugging Face.

In [ ]:
from ttt.models import FullVQAModel

model = FullVQAModel(
    vision_model_name='google/vit-base-patch16-224',
    text_model_name='bert-base-uncased',
    fusion_layers=2,
    fusion_dim=768,
    fusion_heads=12,
    fusion_dropout=0.1,
    pred_hidden=1024,
    pred_dropout=0.2,
    gate_hidden=256,
    gate_dropout=0.1,
    num_answers=N_CLASSES,
).to(device).eval()

n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f'trainable: {n_train:,}  /  total: {n_total:,}')

## 4. Base forward pass and gate confidences

We run the model once and read off gate confidences alongside top-1 predictions. With random weights everything is uninformed; the goal here is simply to verify the wiring.

In [ ]:
with torch.no_grad():
    out = model(
        image=batch['image'].to(device),
        input_ids=batch['input_ids'].to(device),
        attention_mask=batch['attention_mask'].to(device),
    )
logits, conf, z = out['logits'], out['confidence'], out['z']
print('logits :', tuple(logits.shape))
print('conf   :', conf.flatten().detach().cpu().numpy().round(3))
print('z dim  :', tuple(z.shape))
print('top-1  :', logits.argmax(-1).detach().cpu().numpy())

## 5. TTT inner loop with save-and-restore

We invoke `TTTAdapter.adapt_and_predict` on a single 'hard' sample with `K=2` masked-patch steps and verify that the fusion parameters are bit-identical before and after adaptation (the correctness invariant).

In [ ]:
from ttt.ttt_loop import TTTAdapter

adapter = TTTAdapter(
    model=model,
    objective='masked_patch',
    k_steps=2,
    lr=1e-4,
    mask_ratio=0.25,
    grad_clip=1.0,
)

# snapshot fusion params
before = {n: p.detach().clone() for n, p in model.named_parameters() if 'fusion' in n}

i = BATCH - 1  # one of the 'hard' samples
adapt_out = adapter.adapt_and_predict(
    image=batch['image'][i:i+1].to(device),
    input_ids=batch['input_ids'][i:i+1].to(device),
    attention_mask=batch['attention_mask'][i:i+1].to(device),
)
after = {n: p.detach().clone() for n, p in model.named_parameters() if 'fusion' in n}

max_diff = max((before[n] - after[n]).abs().max().item() for n in before)
print(f'adapted top-1     : {adapt_out["logits"].argmax(-1).item()}')
print(f'fusion max |diff| : {max_diff:.2e}   (must be 0.0 — restore invariant)')
assert max_diff == 0.0, 'restore invariant violated'
print('save-and-restore OK')

## 6. Adaptive router and tiny Pareto sweep

We wrap the model in `AdaptiveRouter` and run it across four thresholds. For each threshold we report the skip rate and the average per-sample compute (in GFLOPs). On a real dataset this is the curve `figures/02_solution_pareto_frontier.png` plots; here on 8 random samples the numbers are obviously meaningless, but the routing logic and the FLOPs accounting are exercised end-to-end.

In [ ]:
from ttt.gate import AdaptiveRouter

router = AdaptiveRouter(model=model, adapter=adapter)

rows = []
for tau in [0.0, 0.5, 0.8, 1.0]:
    res = router.route_batch(
        image=batch['image'].to(device),
        input_ids=batch['input_ids'].to(device),
        attention_mask=batch['attention_mask'].to(device),
        threshold=tau,
    )
    skip_rate = float(res['skip_mask'].float().mean().item())
    avg_gflops = float(res.get('avg_gflops', 0.0))
    rows.append((tau, skip_rate, avg_gflops))
    print(f'tau={tau:.2f}   skip={skip_rate*100:5.1f}%   avg_gflops={avg_gflops:6.2f}')

In [ ]:
import matplotlib.pyplot as plt

taus       = [r[0] for r in rows]
skip_rates = [r[1] for r in rows]
gflops     = [r[2] for r in rows]

fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].plot(taus, skip_rates, 'o-'); ax[0].set_xlabel(r'$\tau$'); ax[0].set_ylabel('skip rate')
ax[0].set_title('skip rate vs. threshold')
ax[1].plot(taus, gflops, 'o-', color='C2'); ax[1].set_xlabel(r'$\tau$'); ax[1].set_ylabel('avg GFLOPs/sample')
ax[1].set_title('compute vs. threshold')
plt.tight_layout(); plt.show()

## 7. What this notebook proves

- The model wires up: ViT/BERT/fusion/gate/head produce shape-correct outputs.
- The TTT inner loop runs without errors and **restores parameters bit-exactly** (assertion in cell 5).
- The adaptive router's threshold knob produces a smooth skip-rate / compute trade-off, exactly the mechanism that builds the Pareto frontier on the full dataset.

For the real numbers on VQA-v2 (49.56% base, 49.56% AdaTTT @ τ=0.9, 71.65% Memotion2), see Section IV of the report and the saved `results/` directory in the linked Drive folder.